# 7.12 模型會回答新組合嗎？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：先分整份文件，再切小窗口**

近似改寫的同一家族也放在同一側。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/split.svg")))

**先想一想：**訓練紅圓、藍方，測試紅方，是同一個問題嗎？

把新組合留在訓練之外，比只用沒見過的措辭更能檢查組合規則。先確認每種基礎屬性訓練中都有覆蓋，避免把新詞誤當新組合。

**把直覺寫成數學：**split按完整輸入/屬性組合；不能把同一問答的窗口拆到不同split。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
all_pairs = [(c, s) for c in ("紅", "藍") for s in ("圓", "方")]
held_out = {("紅", "方")}
train = [p for p in all_pairs if p not in held_out]
print("train", train, "held_out", held_out)
assert not set(train) & held_out

**如何讀結果：**例子只是分割；所有可用屬性訓練覆蓋，並不保證模型學到組合。

**只改一件事：**只把整個顏色藍留出，指出現在是在測哪種泛化。
